# Hierarchical Equations of Motion (HEOM) in Quantarhei

This notebook introduces the **Hierarchical Equations of Motion (HEOM)** method
and demonstrates it using the `quantarhei` library on a two-molecule dimer.

**Outline**

1. Why Redfield theory fails at strong system-bath coupling
2. Building a strongly coupled dimer
3. Running HEOM dynamics
4. Plotting the population dynamics
5. Brief comparison with Redfield theory

In [1]:
%config InlineBackend.figure_format = 'svg'

## 1. When Redfield theory is not enough

Standard **Redfield theory** treats the coupling between the electronic system
and its vibrational environment (the *bath*) as a small perturbation.  The
resulting master equation is valid only in the **weak-coupling limit**:

$$\lambda \ll \Delta E, \qquad \lambda \ll k_{\mathrm{B}}T$$

where $\lambda$ is the **reorganisation energy** (half the Stokes shift) and
$\Delta E$ is the relevant electronic energy gap.

When this condition breaks down — as it does in photosynthetic pigment-protein
complexes, conjugated polymers, and other soft-matter chromophore assemblies —
Redfield theory can produce qualitatively wrong dynamics: incorrect relaxation
rates, transiently negative populations, or missing non-Markovian memory.

**HEOM** (Kubo–Tanimura hierarchy) is an *exact* method (within the
Drude–Lorentz bath model) valid for **arbitrary coupling strength**.  It
augments the system density matrix $\rho$ with a hierarchy of auxiliary density
matrices $\rho^{(n)}$ that encode bath memory.  Convergence is controlled by
the **hierarchy depth** $K$: more levels = higher accuracy but higher cost.

| Method | Valid regime | Computational scaling |
|---|---|---|
| Redfield (Markov + secular) | $\lambda \ll J$ | $\mathcal{O}(N^3)$ |
| HEOM at depth $K$ | arbitrary $\lambda$ | $\mathcal{O}(N_{\rm hier}(K)\cdot N^3)$ |

In `quantarhei`, HEOM is provided by `KTHierarchy` (Kubo–Tanimura hierarchy
object) and `KTHierarchyPropagator`.

## 2. Define the model: a strongly coupled dimer

We use the simplest non-trivial system: **two two-level molecules** (a dimer)
with:

* Site-energy difference of 100 cm$^{-1}$ — the two sites are nearly resonant.
* Electronic coupling $J = 100$ cm$^{-1}$ — the excitation delocalises strongly.
* Bath: reorganisation energy $\lambda = 100$ cm$^{-1}$, correlation time
  $\tau_c = 50$ fs, temperature $T = 300$ K.

Because $\lambda \approx J$, we are squarely outside the Redfield validity range.

The bath correlation function is the **high-temperature overdamped Brownian
oscillator**:

$$C(t) = \frac{2\lambda}{\beta\,\tau_c}\,e^{-t/\tau_c}, \qquad \beta = \frac{1}{k_{\mathrm{B}}T}$$

This is the only bath lineshape supported by the current HEOM kernel in
`quantarhei` (`ftype="OverdampedBrownian-HighTemperature"`).

In [2]:
import numpy as np
import quantarhei as qr

# Time axis: 500 steps of 1 fs = 500 fs total
timea = qr.TimeAxis(0.0, 500, 1.0)

# Two two-level molecules
with qr.energy_units("1/cm"):
    m1 = qr.Molecule([0.0, 10000.0])   # lower-energy site
    m2 = qr.Molecule([0.0, 10100.0])   # higher-energy site (initially excited)

# Build dimer aggregate with J = 100 cm-1
agg = qr.Aggregate([m1, m2])
with qr.energy_units("1/cm"):
    agg.set_resonance_coupling(0, 1, 100.0)

# Bath parameters
reorg   = 100  # cm-1  reorganisation energy
cortime =  50  # fs    bath correlation time
T       = 300  # K     temperature

cpar = dict(
    ftype="OverdampedBrownian-HighTemperature",
    reorg=reorg,
    cortime=cortime,
    T=T,
)
with qr.energy_units("1/cm"):
    cfce = qr.CorrelationFunction(timea, cpar)

# Attach the same bath to both molecules
m1.set_transition_environment((0, 1), cfce)
m2.set_transition_environment((0, 1), cfce)

agg.build()

ham = agg.get_Hamiltonian()
sbi = agg.get_SystemBathInteraction()

print("Hamiltonian (cm-1):")
with qr.energy_units("1/cm"):
    print(np.round(ham.data).astype(int))

print(f"\nReorganisation energy : {reorg} cm-1")
print(f"Bath correlation time : {cortime} fs")
print(f"Electronic coupling J : 100 cm-1")
print("=> strong-coupling regime  (lambda ~ J)")

Hamiltonian (cm-1):
[[    0     0     0]
 [    0 10000   100]
 [    0   100 10100]]

Reorganisation energy : 100 cm-1
Bath correlation time : 50 fs
Electronic coupling J : 100 cm-1
=> strong-coupling regime  (lambda ~ J)


## 3. Build the HEOM hierarchy and propagate

We create two `KTHierarchy` objects at depth $K = 3$ and $K = 4$ to verify
convergence — if the two results agree, depth 3 is sufficient.

**Index convention in `quantarhei`**: the aggregate Hamiltonian always includes
the collective ground state at index 0.  Single-exciton states start at index 1.
For our dimer: index 1 = molecule 1 excited, index 2 = molecule 2 excited.

**Initial condition**: molecule 2 fully excited ($\rho_{22}(0) = 1$), mimicking
instantaneous photoexcitation of the higher-energy site.

In [3]:
import time as _time

# Build hierarchies at two depths
Hy3 = qr.KTHierarchy(ham, sbi, 3)
Hy4 = qr.KTHierarchy(ham, sbi, 4)
print(f"Hierarchy depth {Hy3.depth} — auxiliary matrices: {Hy3.hsize}")
print(f"Hierarchy depth {Hy4.depth} — auxiliary matrices: {Hy4.hsize}")

# Initial density matrix — all population on site 2 (index 2)
rhoi = qr.ReducedDensityMatrix(dim=ham.dim)
rhoi.data[2, 2] = 1.0

# Propagators
kprop3 = qr.KTHierarchyPropagator(timea, Hy3)
kprop4 = qr.KTHierarchyPropagator(timea, Hy4)

# Propagate — report_hierarchy=False suppresses per-step prints
rhot3 = kprop3.propagate(rhoi, report_hierarchy=False, free_hierarchy=False)
print("HEOM depth 3 propagated")

rhot4 = kprop4.propagate(rhoi, report_hierarchy=False, free_hierarchy=False)
print("HEOM depth 4 propagated")

Hierarchy depth 3 — auxiliary matrices: 10
Hierarchy depth 4 — auxiliary matrices: 15


HEOM depth 3 propagated


HEOM depth 4 propagated


## 4. Population dynamics

We plot $\rho_{11}(t)$ (site 1, blue) and $\rho_{22}(t)$ (site 2, red) in the
**site basis**.  Solid lines = depth 3, dashed lines = depth 4.  Close overlap
confirms convergence.

In [4]:
import matplotlib.pyplot as plt

N = timea.length
t = timea.data

fig, ax = plt.subplots(figsize=(8, 5))

# Depth-3 HEOM (solid lines)
ax.plot(t[:N], rhot3.data[:N, 1, 1], color="C0", lw=2,       label=r"site 1  $\rho_{11}$ (K=3)")
ax.plot(t[:N], rhot3.data[:N, 2, 2], color="C3", lw=2,       label=r"site 2  $\rho_{22}$ (K=3)")

# Depth-4 HEOM (dashed lines) — convergence check
ax.plot(t[:N], rhot4.data[:N, 1, 1], color="C0", lw=1.5, ls="--", label="site 1  (K=4)")
ax.plot(t[:N], rhot4.data[:N, 2, 2], color="C3", lw=1.5, ls="--", label="site 2  (K=4)")

ax.set_xlabel("Time (fs)", fontsize=12)
ax.set_ylabel("Population", fontsize=12)
ax.set_title(
    r"HEOM population dynamics — dimer, $\lambda = J = 100\,{\rm cm}^{-1}$, $T=300\,{\rm K}$",
    fontsize=11,
)
ax.set_xlim(0, 500)
ax.set_ylim(-0.05, 1.05)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

/Users/danherma/projects-personal/quantarhei/.venv/lib/python3.10/site-packages/matplotlib/cbook.py:1719: ComplexWarning: Casting complex values to real discards the imaginary part
  return math.isfinite(val)
/Users/danherma/projects-personal/quantarhei/.venv/lib/python3.10/site-packages/matplotlib/cbook.py:1355: ComplexWarning: Casting complex values to real discards the imaginary part
  return np.asarray(x, float)
/var/folders/vx/gjt_y6_50z3b41b0w7yk4ryc0000gn/T/ipykernel_61799/1188625668.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**What to notice**

* Population flows from the initially excited site 2 to site 1 on a ~100–300 fs
  timescale.
* **Quantum beats** (oscillations) appear on top of the incoherent relaxation;
  they arise from electronic coherence driven by the inter-site coupling $J$
  and are damped by the bath.
* Depth-3 and depth-4 curves virtually coincide, confirming that $K = 3$ is
  converged at these parameters.
* At long times the population ratio reflects the Boltzmann factor for the
  site-energy gap — site 1 ends up more populated (it is lower in energy).

## 5. Comparison with Redfield theory

We run the **secular Redfield** master equation on the same system.  Because
$\lambda \approx J$, Redfield is outside its validity range and we expect
systematic differences — typically smoother (over-)estimated relaxation and
suppressed quantum beats.

`quantarhei` provides Redfield through
`Aggregate.get_ReducedDensityMatrixPropagator` with
`relaxation_theory="standard_Redfield"`.  The propagator integrates

$$\dot{\rho}(t) = -\frac{i}{\hbar}[H,\rho(t)] + \mathcal{R}\,\rho(t)$$

where $\mathcal{R}$ is the time-independent (Markovian) Redfield tensor.

In [5]:
# Redfield propagator via the aggregate convenience method
prop_rf = agg.get_ReducedDensityMatrixPropagator(
    timea,
    relaxation_theory="standard_Redfield",
    time_dependent=False,
    secular_relaxation=True,
)

# Same initial condition as HEOM
rhoi_rf = qr.ReducedDensityMatrix(dim=ham.dim)
rhoi_rf.data[2, 2] = 1.0

rhot_rf = prop_rf.propagate(rhoi_rf)
print("Redfield propagation done.")

Redfield propagation done.


In [6]:
fig, ax = plt.subplots(figsize=(8, 5))

# HEOM depth-3 (solid)
ax.plot(t[:N], rhot3.data[:N, 1, 1], color="C0", lw=2,       label="site 1  HEOM (K=3)")
ax.plot(t[:N], rhot3.data[:N, 2, 2], color="C3", lw=2,       label="site 2  HEOM (K=3)")

# Redfield (dashed)
ax.plot(t[:N], rhot_rf.data[:N, 1, 1], color="C0", lw=1.5, ls="--", label="site 1  Redfield")
ax.plot(t[:N], rhot_rf.data[:N, 2, 2], color="C3", lw=1.5, ls="--", label="site 2  Redfield")

ax.set_xlabel("Time (fs)", fontsize=12)
ax.set_ylabel("Population", fontsize=12)
ax.set_title(
    r"HEOM vs Redfield — dimer, $\lambda = J = 100\,{\rm cm}^{-1}$, $T=300\,{\rm K}$",
    fontsize=11,
)
ax.set_xlim(0, 500)
ax.set_ylim(-0.05, 1.05)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

/var/folders/vx/gjt_y6_50z3b41b0w7yk4ryc0000gn/T/ipykernel_61799/1008658875.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Interpreting the comparison**

In the strong-coupling regime ($\lambda \approx J$) the two methods disagree:

* **Redfield** (dashed) typically produces smoother, faster-relaxing curves
  because the secular approximation averages away coherence contributions.
* **HEOM** (solid) retains non-Markovian bath memory and inter-site quantum
  coherence, producing visible oscillations at short times.
* Both methods reach the same **thermal equilibrium** at long times (correct
  Boltzmann population ratio), but the transient pathways differ significantly.

> **Practical rule**: for molecular aggregates with $\lambda \gtrsim 50$ cm$^{-1}$
> and $J \sim \lambda$, always benchmark against HEOM before trusting Redfield
> results — especially for early-time coherent dynamics.

## Summary

| Task | Quantarhei object / call |
|---|---|
| Bath correlation function | `qr.CorrelationFunction(timea, {"ftype": "OverdampedBrownian-HighTemperature", "reorg": ..., "cortime": ..., "T": ...})` |
| Build the hierarchy | `qr.KTHierarchy(ham, sbi, depth)` |
| Create HEOM propagator | `qr.KTHierarchyPropagator(timea, Hy)` |
| Run propagation | `rhot = kprop.propagate(rhoi, report_hierarchy=False, free_hierarchy=False)` |
| Access populations | `rhot.data[time_index, site_i, site_i]` |
| Redfield reference | `agg.get_ReducedDensityMatrixPropagator(timea, relaxation_theory="standard_Redfield", ...)` |

**Further reading**

* Kubo & Tanimura, *J. Phys. Soc. Japan* **58**, 101 (1989) — original HEOM paper.
* Ishizaki & Fleming, *J. Chem. Phys.* **130**, 234111 (2009) — HEOM applied to
  photosynthetic energy transfer.
* `examples/demo_013_HEOM.py` in this repository — three-molecule trimer with
  hierarchy convergence comparison and memory-kernel extraction.